In [2]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset,DataLoader
import torch.distributed as dist
from datetime import timedelta
import torch.nn.functional as F
import numpy as np
import os
import random
import logging

logging.basicConfig(filename='/app/logit.log', level=logging.INFO,format='%(asctime)s %(message)s')

In [3]:
def init(rank, world_size, backend='gloo'):
    os.environ['GLOO_SOCKET_IFNAME'] = 'eth0'
    os.environ['MASTER_ADDR'] = 'c1'
    os.environ['MASTER_PORT'] = '29500'

    dist.init_process_group(
        backend=backend,
        world_size=world_size,
        rank=rank,
        timeout=timedelta(seconds=60),
    )

    print(f"Rank {rank}: is_initialized and ready to communicate...")
    return dist.is_initialized()

def recv(arr):
    dist.recv(tensor=arr, src=0)

def send(arr):
    dist.send(tensor=arr, dst=0)

In [4]:
class TrainDataset(Dataset):
    def __init__(self,transform=None,path='/app/y_train.npy'):
        self.y= np.load(path)
        self.n = self.y.shape[0]
        self.transform =transform
        
    def __getitem__(self, index):
        sample_temp = self.y[index]
        
        if self.transform:
            sample_temp = self.transform(sample_temp)
            
        sample = sample_temp,index
        return sample
    
    def __len__(self):
        return self.n
    
    def shape(self):
        return self.n,self.height,self.width
    

class ToTensor:
    def __call__(self,input):
        input = input.reshape(1)
        return torch.from_numpy(input)
            

In [5]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        #c2
        self.conv2 = nn.Conv2d(10, 20, kernel_size=5)
        self.conv2_drop = nn.Dropout2d()
        self.fc1 = nn.Linear(320, 50)
        self.fc2 = nn.Linear(50, 10)

    def forward(self, x):
        x = F.relu(F.max_pool2d(self.conv2_drop(self.conv2(x)), 2))
        x = x.view(-1, 320)
        x = F.relu(self.fc1(x))
        x = F.dropout(x, training=self.training)
        x = self.fc2(x)
        return x

In [6]:

batch_size = 250
criterion = nn.CrossEntropyLoss()
transform = ToTensor()
dataset = TrainDataset(transform=transform)
dataloader = DataLoader(dataset=dataset,shuffle=False,batch_size=batch_size)

random.seed(43)
np.random.seed(43)
torch.manual_seed(43)

model =CNN()
optim = torch.optim.Adam(params=model.parameters(),lr=0.005)

saved_model_dict = torch.load("/app/model.pth")
filter_layer ={layer_name:params for layer_name,params in saved_model_dict.items() if not layer_name.startswith('conv1')}
model.load_state_dict(filter_layer)
model.eval()

CNN(
  (conv2): Conv2d(10, 20, kernel_size=(5, 5), stride=(1, 1))
  (conv2_drop): Dropout2d(p=0.5, inplace=False)
  (fc1): Linear(in_features=320, out_features=50, bias=True)
  (fc2): Linear(in_features=50, out_features=10, bias=True)
)

In [7]:
def run(criterion=criterion,send=send,recv=recv,num_epoch=10, batch_size=1, num_classes=10):
    for epoch in range(num_epoch):
        for i, (target, index) in enumerate(dataloader):
        
            smashed_data = torch.zeros(size=(batch_size,10,12,12),requires_grad=True)
            recv(smashed_data)
            
            logits = model(smashed_data)
            # logits_np = logits.detach().cpu().numpy()
            # logging.info(f"Epoch {epoch+1}, Batch {i+1}: Logits:\n{logits_np}")
            loss = criterion(logits,target.view(-1))
            
            loss.backward()

            gradient = smashed_data.grad
            send(gradient)
            
            optim.step()
            optim.zero_grad()

            
            
            print(f"epoch {epoch} {i}/{len(dataloader)} loss : {loss}")
            if i == len(dataloader)-1:
                logging.info(f"Epoch {epoch+1}, Batch {i+1}: loss= { loss}")
                
                
                

In [8]:
num_epoch = 1
criterion = nn.CrossEntropyLoss()
num_classes = 10
init(1,2)
run(batch_size=batch_size,num_epoch=num_epoch)

dist.destroy_process_group()


KeyboardInterrupt: 

In [ ]:
for param_group in optim.param_groups:
        param_group['params'] = list(model.parameters())

NameError: name 'optim' is not defined